[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch13-model-serving/02-gom-batch.ipynb)

# Gom batch: đổi độ trễ lấy thông lượng

Chương 13 gọi gom batch là đòn bẩy cốt lõi của kinh tế phục vụ: cùng một GPU, xử lý 32 ảnh một lần
thay vì từng ảnh, cho thông lượng gấp nhiều lần, vì chi phí cố định được chia cho nhiều yêu cầu.
Cái giá là thời gian chờ: một yêu cầu phải đợi những yêu cầu khác tới để cùng vào batch. Chương gọi
đó là thuế gom batch, và dành cả một mục để tính nó.

Sổ tay này tính lại các bảng của mục ấy, rồi dựng một bộ lập lịch gom batch thật bằng mô phỏng để
xem những công thức trung bình của chương đứng ở đâu so với phần đuôi.

**Bạn sẽ làm:**
1. tính hai cột suy ra của bảng 9, rồi đo trên CPU của bạn hai chi phí cố định mà gom batch chia nhỏ;
2. tính thuế gom batch của phương trình 4 và kiểm nó bằng mô phỏng gom batch tĩnh;
3. mô phỏng cửa sổ gom batch theo lịch cố định, đặt kết quả cạnh phương trình 8, bảng 10 và
   phương trình 13;
4. tính bảng 11, ví dụ 1.7 và bảng 12 từ mô hình thời gian phục vụ của phương trình 10;
5. mô phỏng năm cấu hình gom batch của bảng 13 và vẽ biên Pareto;
6. tính cấu hình thực tế của bảng 14 và quy tắc cửa sổ của phương trình 15;
7. cho một đợt tăng vọt đi qua batch 1 và qua gom batch động;
8. tính hai dạng lưu lượng khác: sáu camera với một hạn chót, và một người dùng trên điện thoại.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # thứ tự màu cố định

# Mô hình thời gian phục vụ của chương cho ResNet-50 trên V100 (phương trình 10 và bảng 11)
PHU_TROI = sach(5, nguon="5 ms · 0.6 ms") / 1000        # giây, phụ trội cố định mỗi batch
MOI_ANH = sach(0.6, nguon="5 ms · 0.6 ms") / 1000       # giây cho mỗi ảnh trong batch


def t_svc(B):
    """Thời gian phục vụ một batch B ảnh, tính bằng giây."""
    return PHU_TROI + MOI_ANH * B


def den_poisson(lam, n, rng):
    """n thời điểm đến theo Poisson với lam yêu cầu mỗi giây."""
    return np.cumsum(rng.exponential(1 / lam, n))


def gom_dong(den, W, B_max, svc=t_svc):
    """Gom batch động: cửa sổ mở khi yêu cầu đầu tiên đang chờ tới, và batch được gửi khi đủ B_max
    yêu cầu hoặc khi hết W giây, nhưng chỉ khi GPU đã rảnh.
    Trả về độ trễ của từng yêu cầu (giây) và kích thước batch mà nó nằm trong."""
    n = len(den)
    tre, thay = np.empty(n), np.empty(n, dtype=int)
    i, ranh = 0, 0.0
    while i < n:
        j = i + B_max - 1
        luc_day = den[j] if j < n else np.inf        # thời điểm batch đầy
        gui = max(ranh, min(den[i] + W, luc_day))
        k = min(B_max, int(np.searchsorted(den, gui, side="right")) - i)
        xong = gui + svc(k)
        tre[i:i + k] = xong - den[i:i + k]
        thay[i:i + k] = k
        ranh, i = xong, i + k
    return tre, thay


def gom_tinh(den, B, svc=t_svc):
    """Gom batch tĩnh: chờ đủ đúng B yêu cầu rồi mới gửi."""
    n = (len(den) // B) * B
    tre = np.empty(n)
    ranh = 0.0
    for i in range(0, n, B):
        gui = max(ranh, den[i + B - 1])
        xong = gui + svc(B)
        tre[i:i + B] = xong - den[i:i + B]
        ranh = xong
    return tre


def gom_theo_lich(den, W, svc=t_svc):
    """Cửa sổ lịch cố định: cửa sổ thứ k đóng ở (k + 1)·W. Khi cửa sổ đóng, mọi yêu cầu đã tới thành
    một batch; nếu GPU còn bận, batch đi ngay khi GPU rảnh và mang theo cả những yêu cầu tới trong lúc ấy.
    Trả về độ trễ, thời gian chờ và kích thước batch mà mỗi yêu cầu thấy."""
    n = len(den)
    tre, cho, thay = np.empty(n), np.empty(n), np.empty(n, dtype=int)
    i, ranh = 0, 0.0
    while i < n:
        dong = (math.floor(den[i] / W) + 1) * W
        gui = max(dong, ranh)
        j = int(np.searchsorted(den, gui, side="right"))
        xong = gui + svc(j - i)
        tre[i:j], cho[i:j], thay[i:j] = xong - den[i:j], gui - den[i:j], j - i
        ranh, i = xong, j
    return tre, cho, thay


def poisson_pmf(k, m):
    return math.exp(-m) * m ** k / math.factorial(k)


def poisson_cdf(k, m):
    return sum(poisson_pmf(j, m) for j in range(k + 1))


def poisson_ppf(q, m):
    k = 0
    while poisson_cdf(k, m) < q:
        k += 1
    return k


def thoi_gian(f, lap=60, khoi_dong=5):
    """Trung vị thời gian của f() qua nhiều lần chạy, tính bằng giây."""
    for _ in range(khoi_dong):
        f()
    ts = []
    for _ in range(lap):
        t1 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t1)
    return float(np.median(ts))


print("đã nạp t_svc(B) = 5 ms + 0.6 ms · B và ba bộ lập lịch: gom_tinh, gom_theo_lich, gom_dong")

## 1. Vì sao gom batch có lợi

Napkin math 1.4 quét kích thước batch từ 1 tới 32 cho ResNet-50 trên V100. Sách nói rõ kích thước
batch, thời gian suy luận và mức sử dụng GPU là giả định của kịch bản; hai cột còn lại, thời gian
tính mỗi ảnh và thông lượng, được suy ra bằng phép chia. Ô dưới làm hai phép chia ấy.

In [ ]:
BANG9 = {  # kích thước batch: thời gian suy luận (ms), giả định của kịch bản
    1: sach(5, nguon="32 · 50 · 5 ms · 200 ảnh"),
    4: sach(7.2, nguon="7.2 ms · 556 ảnh"),
    8: sach(9.1, nguon="9.1 ms · 879 ảnh"),
    16: sach(14, nguon="16 · 14 ms · 1,143 ảnh"),
    32: sach(25, nguon="25 ms · 1,280 ảnh"),
}
print(" batch   suy luận   mỗi ảnh   thông lượng")
for B, ms in BANG9.items():
    print(f"{B:>6} {ms:8.1f} ms {ms / B:7.2f} ms {1000 * B / ms:9.0f} ảnh/s")

theo_sach("batch 4, mỗi ảnh (ms)", BANG9[4] / 4, sach=1.8, trich="4 | 7.2 ms | 1.8 ms | 556 img/s | 42%")
theo_sach("batch 8, mỗi ảnh (ms)", BANG9[8] / 8, sach=1.1, nguon="8 ảnh · 1.1 ms")
theo_sach("batch 16, mỗi ảnh (ms)", BANG9[16] / 16, sach=0.9, trich="16 | 14 ms | 0.9 ms | 1,143 img/s | 85%")
theo_sach("batch 32, mỗi ảnh (ms)", BANG9[32] / 32, sach=0.8, trich="32 | 25 ms | 0.8 ms | 1,280 img/s | 95%")
theo_sach("batch 1, thông lượng (ảnh/s)", 1000 / BANG9[1], sach=200, nguon="15 % · 200 ảnh")
theo_sach("batch 4, thông lượng (ảnh/s)", 1000 * 4 / BANG9[4], sach=556, nguon="7.2 ms · 556 ảnh")
theo_sach("batch 8, thông lượng (ảnh/s)", 1000 * 8 / BANG9[8], sach=879, nguon="9.1 ms · 879 ảnh")
theo_sach("batch 16, thông lượng (ảnh/s)", 1000 * 16 / BANG9[16], sach=1143, nguon="16 · 14 ms · 1,143 ảnh")
theo_sach("batch 32, thông lượng (ảnh/s)", 1000 * 32 / BANG9[32], sach=1280, nguon="25 ms · 1,280 ảnh")
theo_sach("batch 32 so với batch 1 (lần)", (32 / BANG9[32]) / (1 / BANG9[1]), sach=6.4, nguon="1,280 ảnh · 6.4 lần")

Thời gian suy luận tăng 5 lần (5 ms lên 25 ms) trong khi số ảnh tăng 32 lần, nên mỗi ảnh rẻ đi
hơn sáu lần. Nhưng 25 ms đó mới là thời gian suy luận thuần. Một yêu cầu còn phải chờ cửa sổ gom
batch đóng lại. Sách tính hai trường hợp:

In [ ]:
CUA_10 = sach(10, nguon="10 ms · 35 ms")
theo_sach("cửa sổ 10 ms + batch 32 (ms)", CUA_10 + BANG9[32], sach=35, nguon="10 ms · 35 ms")
CUA_5 = sach(5, trich="worst-case user latency of about 14 ms (5 ms wait plus 9 ms inference)")
theo_sach("cửa sổ 5 ms + batch 8, xấu nhất (ms)", CUA_5 + BANG9[8], sach=14,
          trich="worst-case user latency of about 14 ms (5 ms wait plus 9 ms inference)")
NGAN_SACH = sach(20, nguon="35 ms · 20 ms")
print(f"dưới ngân sách {NGAN_SACH} ms: {'có' if CUA_5 + BANG9[8] <= NGAN_SACH else 'không'}")
theo_sach("batch 8 so với batch 1, thông lượng (lần)", (8 / BANG9[8]) / (1 / BANG9[1]), sach=4.4,
          nguon="20 ms · 4.4 lần")

Sách thêm một điều kiện: tải phải đủ cao để lấp đầy cửa sổ. Phần 3 sẽ cho thấy điều kiện ấy quan
trọng thế nào.

### Hai chi phí cố định

Ở batch nhỏ, hai chi phí cố định chiếm phần lớn. Thứ nhất là phụ trội khởi chạy kernel: mỗi tầng
cần một lần CPU chuẩn bị tham số và báo GPU bắt đầu, thường 5 tới 20 μs mỗi kernel, không phụ
thuộc kích thước batch. Thứ hai là đọc trọng số: ở batch 1, GPU đọc hết trọng số cho một ảnh; ở
batch 32, cùng một lần đọc phục vụ 32 ảnh.

In [ ]:
KERNEL_IT = sach(5, trich="typically 5–20 μs per kernel")
KERNEL_NHIEU = sach(20, nguon="20 μs · 50")
SO_TANG = sach(50, nguon="20 μs · 50")
theo_sach("khởi chạy kernel, ít nhất (μs)", SO_TANG * KERNEL_IT, sach=250, nguon="50 · 250")
theo_sach("khởi chạy kernel, nhiều nhất (μs)", SO_TANG * KERNEL_NHIEU, sach=1000, nguon="250 · 1000 μs")
theo_sach("hiệu quả bộ nhớ ở batch 32 (lần)", 32 / 1, sach=32, nguon="32 · 32 lần")
print(f"chia cho 32 ảnh của một batch: {SO_TANG * KERNEL_IT / 32:.1f} tới {SO_TANG * KERNEL_NHIEU / 32:.1f} μs mỗi ảnh")

CPU không có kernel GPU, nhưng nó có một chi phí cùng loại: mỗi lần PyTorch gọi một tầng, khung
chạy phải kiểm kiểu dữ liệu, chọn hàm tính và cấp phát đầu ra, bất kể tầng ấy nhỏ hay lớn. Ô dưới
đo chi phí đó bằng một mô hình gồm toàn những tầng `Linear(16, 16)`, nhỏ tới mức phần tính gần như
bằng không, rồi đếm thời gian theo số tầng. Độ dốc của đường thẳng là chi phí cố định của một lần
gọi.

**Dự đoán:** trên CPU của bạn, một mạng 50 tầng như thế mất bao lâu ở batch 1: vài micro giây, vài
trăm micro giây, hay vài mili giây?

In [ ]:
torch.set_num_threads(1)
cac_so_tang = [1, 10, 25, 50, 100]
tg_tang = []
with torch.inference_mode():
    for d in cac_so_tang:
        mang = torch.nn.Sequential(*[torch.nn.Linear(16, 16) for _ in range(d)]).eval()
        x1 = torch.randn(1, 16)
        tg_tang.append(thoi_gian(lambda: mang(x1), lap=200))
moi_tang, _ = np.polyfit(cac_so_tang, tg_tang, 1)
for d, t in zip(cac_so_tang, tg_tang):
    do_tren_may(f"{d:>3} tầng Linear(16, 16), batch 1", 1e6 * t, "μs")
do_tren_may("độ dốc: chi phí cố định của một lần gọi tầng", 1e6 * moi_tang, "μs")

Giờ đến chi phí thứ hai, đo theo kích thước batch. Mô hình là một perceptron nhiều tầng nhỏ (784 →
256 → 256 → 256 → 256 → 10, giả định của sổ tay, cỡ một bộ phân loại MNIST), chạy trên một luồng
CPU. Nếu thời gian một batch có dạng $a + bB$ như mô hình của chương, thì $a$ là phần cố định mà
batch chia nhỏ, còn $b$ là phần mỗi mẫu không chia được.

**Dự đoán:** batch 256 cho thông lượng gấp bao nhiêu lần batch 1 trên CPU của bạn?

In [ ]:
mlp = torch.nn.Sequential(
    torch.nn.Linear(784, 256), torch.nn.ReLU(), torch.nn.Linear(256, 256), torch.nn.ReLU(),
    torch.nn.Linear(256, 256), torch.nn.ReLU(), torch.nn.Linear(256, 256), torch.nn.ReLU(),
    torch.nn.Linear(256, 10)).eval()
CAC_B = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256])
tg_batch = []
with torch.inference_mode():
    for B in CAC_B:
        xb = torch.randn(int(B), 784)
        tg_batch.append(thoi_gian(lambda: mlp(xb), lap=60))
tg_batch = np.array(tg_batch)
b_do, a_do = np.polyfit(CAC_B, tg_batch, 1)
do_tren_may("a, phần cố định mỗi batch (khớp đường thẳng)", 1000 * a_do, "ms")
do_tren_may("b, phần mỗi mẫu (khớp đường thẳng)", 1000 * b_do, "ms")
do_tren_may("phần cố định trong một batch 1, theo đường khớp", 100 * a_do / (a_do + b_do), "%")
do_tren_may("phần cố định trong một batch 256, theo đường khớp", 100 * a_do / (a_do + 256 * b_do), "%")
do_tren_may("thông lượng batch 256 so với batch 1", (256 / tg_batch[-1]) / (1 / tg_batch[0]), "lần")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(CAC_B, 1000 * tg_batch, "o", color=MAU[0], ms=7, mec="white", mew=1.2, label="đo")
ax1.plot(CAC_B, 1000 * (a_do + b_do * CAC_B), color=MAU[1], lw=2, label="khớp a + b·B")
ax1.set_xlabel("kích thước batch B")
ax1.set_ylabel("thời gian một batch (ms)")
ax1.set_title("Thời gian một batch (đo trên máy này)")
ax1.legend(loc="upper left", fontsize=8)
ax2.semilogx(CAC_B, CAC_B / tg_batch, "o-", color=MAU[0], lw=2, ms=7, mec="white", mew=1.2)
ax2.set_xscale("log", base=2)
ax2.set_xticks(CAC_B, [str(B) for B in CAC_B])
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax2.set_xlabel("kích thước batch B (thang log)")
ax2.set_ylabel("thông lượng (mẫu mỗi giây)")
ax2.set_title("Thông lượng theo batch (đo trên máy này)")
ax2.set_ylim(0, None)
plt.tight_layout()
plt.show()

Đọc hai biểu đồ cùng nhau. Bên trái, thời gian một batch tăng chậm hơn nhiều so với số mẫu: batch
256 có số mẫu gấp 256 lần batch 1 mà không tốn gấp 256 lần thời gian. Đường thẳng khớp không đi qua
gốc, và phần chặn của nó là chi phí trả một lần mỗi batch, thứ mà batch chia nhỏ. Bên phải là cùng
điều ấy, đọc theo thông lượng. Đó cũng là hình dạng của bảng 9, dù con số của CPU và của V100 khác
nhau hoàn toàn.

Các điểm đo thường không nằm thẳng hàng. Thư viện tính toán có đường đi riêng cho phép nhân ma trận
với một vectơ (batch 1), và bộ nhớ đệm của CPU đổi vai trò theo kích thước ma trận. Chương nói đúng
điều này về mô hình 5 + 0.6B: nó nắm xu hướng chính, còn thời gian phục vụ thật lệch khỏi nó vì hệ
thống phân cấp bộ nhớ.

## 2. Thuế gom batch

Với gom batch theo kích thước cố định, chương tách độ trễ thành hai nguồn. Thứ nhất là thời gian
chờ đủ batch: yêu cầu tới đầu chờ $B-1$ yêu cầu khác, còn yêu cầu tới cuối không chờ gì. Thứ hai là
thời gian suy luận phình ra, vì GPU xử lý $B$ mẫu thay vì một. Với yêu cầu đến theo Poisson,
phương trình 4 cho

$$L_\text{lat,total} \approx \frac{B-1}{2\lambda_\text{arr}} + T_\text{inf}(B).$$

In [ ]:
LAM_4 = sach(500, nguon="500 QPS · 32 · 0")


def cho_gom(B, lam):
    return (B - 1) / (2 * lam)


theo_sach("chờ gom ở batch 1 (ms)", 1000 * cho_gom(1, LAM_4), sach=0, nguon="500 QPS · 32 · 0")
theo_sach("chờ gom ở batch 32 (ms)", 1000 * cho_gom(32, LAM_4), sach=31, nguon="0 · 31 ms")
T1_4 = sach(2, trich="(2 ms → 46 ms)")              # độ trễ tổng ở batch 1, tức T_inf(1)
TONG32_4 = sach(46, nguon="31 ms · 46 ms")
theo_sach("độ trễ tổng tăng (lần)", TONG32_4 / T1_4, sach=23, nguon="46 ms · 23 lần")
print(f"suy ra: T_inf(32) của ví dụ này là {TONG32_4 - 1000 * cho_gom(32, LAM_4):.0f} ms")

Ví dụ của phương trình 4 không nói $T_\text{inf}(B)$ đổi theo $B$ ra sao, ngoài hai đầu mút. Để mô
phỏng, sổ tay dùng mô hình thời gian phục vụ mà chương dùng ở phương trình 10, $5 + 0.6B$ ms, và cho
yêu cầu đến theo Poisson tới một bộ lập lịch chờ đủ đúng 32 yêu cầu.

**Dự đoán:** ở 500 QPS, độ trễ trung bình mô phỏng có khớp phương trình 4 không? Và nếu lưu lượng
chỉ còn 50 QPS, yêu cầu chờ trung bình bao lâu để đủ batch?

In [ ]:
for lam in (LAM_4, 50):
    tre_tinh = gom_tinh(den_poisson(lam, 100_000, np.random.default_rng(42)), 32)
    print(f"{lam:>4} QPS, gom batch tĩnh 32: trung bình {1000 * tre_tinh.mean():6.1f} ms"
          f" (phương trình 4: {1000 * (cho_gom(32, lam) + t_svc(32)):6.1f} ms),"
          f" p99 {1000 * np.percentile(tre_tinh, 99):6.1f} ms, lâu nhất {1000 * tre_tinh.max():6.0f} ms")

Trung bình mô phỏng khớp phương trình 4, vì phương trình ấy chỉ bỏ qua hàng đợi khi GPU còn bận,
và ở đây GPU rảnh trước khi batch kế tiếp đủ. Nhưng p99 cao hơn trung bình nhiều: yêu cầu tới đầu
batch phải chờ cả 31 yêu cầu sau, chứ không phải một nửa. Ở 50 QPS, chờ
trung bình đã lên hàng trăm mili giây. Đây là điểm yếu của gom batch tĩnh mà chương nêu: khi lưu
lượng thấp, yêu cầu có thể chờ rất lâu cho đủ batch. Gom batch động sửa nó bằng một cửa sổ thời
gian có giới hạn.

## 3. Cửa sổ gom batch theo lịch cố định

Phương trình 7 tách độ trễ thành thời gian chờ trong hàng đợi gom batch cộng thời gian tính của
batch. Với cửa sổ lịch cố định (cửa sổ đóng đều đặn mỗi $T_\text{window}$) và yêu cầu đến theo
Poisson, thời điểm đến phân bố đều trong cửa sổ, nên phương trình 8 cho thời gian chờ trung bình
bằng nửa cửa sổ.

In [ ]:
CUA_20 = sach(20, nguon="20 ms · 10 ms · 50 ms") / 1000
SLO_8 = sach(50, nguon="10 ms · 50 ms · 5 ms")
theo_sach("chờ trung bình với cửa sổ 20 ms (ms)", 1000 * CUA_20 / 2, sach=10, nguon="20 ms · 10 ms · 50 ms")
theo_sach("phần ngân sách SLO trung bình (%)", 100 * (1000 * CUA_20 / 2) / SLO_8, sach=20, nguon="5 ms · 20 %")

den_20 = den_poisson(200, 200_000, np.random.default_rng(42))       # 200 QPS, giả định
cho_20 = (np.floor(den_20 / CUA_20) + 1) * CUA_20 - den_20
print(f"mô phỏng 200,000 yêu cầu: chờ trung bình {1000 * cho_20.mean():.2f} ms, lâu nhất {1000 * cho_20.max():.2f} ms")

Chờ trung bình đúng nửa cửa sổ, và chờ lâu nhất đúng bằng cả cửa sổ, cho yêu cầu tới ngay khi
cửa sổ vừa mở. Vì vậy chương dặn: SLO theo trung bình tiêu nửa cửa sổ, còn SLO theo phần đuôi phải
tính cả cửa sổ.

Số yêu cầu gom được trong một cửa sổ cũng ngẫu nhiên. Phương trình 9 nói nó theo phân phối Poisson
với trung bình $\lambda_\text{arr} T_\text{window}$, và phương sai bằng trung bình. Bảng 10 đọc
phương trình ấy ở bốn mức lưu lượng với cửa sổ 10 ms.

**Dự đoán:** ở 50 QPS, bao nhiêu phần trăm cửa sổ không có yêu cầu nào, tức không có batch nào để
chạy?

In [ ]:
W10 = sach(10, nguon="10 ms · 50 QPS") / 1000
MUC_10 = [50, 200, 500, 1000]                      # QPS, bốn dòng của bảng 10
dem = {}
for lam in MUC_10:
    den_l = den_poisson(lam, 400_000, np.random.default_rng(42))
    o = np.floor(den_l / W10).astype(int)
    dem[lam] = np.bincount(o)[:-1]                  # bỏ cửa sổ cuối, chưa đóng
print("  QPS  TB(pt.9)  TB(mp)  ĐLC(pt.9)  ĐLC(mp)  P(0) pt.9   P(0) mp  P(≥2TB) pt.9  P(≥2TB) mp")
cong_thuc = {}
for lam in MUC_10:
    m = lam * W10
    p0 = poisson_pmf(0, m)
    p2 = 1 - poisson_cdf(math.ceil(2 * m) - 1, m)
    cong_thuc[lam] = (m, math.sqrt(m), 100 * p0, 100 * p2)
    c = dem[lam]
    print(f"{lam:>5} {m:8.2f} {c.mean():7.2f} {math.sqrt(m):9.2f} {c.std():8.2f} {100 * p0:9.3f} % {100 * np.mean(c == 0):7.3f} %"
          f" {100 * p2:10.2f} % {100 * np.mean(c >= 2 * m):9.2f} %")

theo_sach("50 QPS: batch trung bình", cong_thuc[50][0], sach=0.5, nguon="50 QPS · 0.5")
theo_sach("50 QPS: độ lệch chuẩn", cong_thuc[50][1], sach=0.7, trich="50 QPS | 0.5 | 0.7 | 61% | 39%")
theo_sach("50 QPS: cửa sổ rỗng (%)", cong_thuc[50][2], sach=61, nguon="0.5 · 61 %")
theo_sach("50 QPS: batch ≥ 2 lần trung bình (%)", cong_thuc[50][3], sach=39, nguon="0.3 % · 39 %")
theo_sach("200 QPS: batch trung bình", cong_thuc[200][0], sach=2, trich="200 QPS | 2 | 1.4 | 14% | 14%")
theo_sach("200 QPS: độ lệch chuẩn", cong_thuc[200][1], sach=1.4, trich="200 QPS | 2 | 1.4 | 14% | 14%")
theo_sach("200 QPS: cửa sổ rỗng (%)", cong_thuc[200][2], sach=14, nguon="200 QPS · 14 %")
theo_sach("200 QPS: batch ≥ 2 lần trung bình (%)", cong_thuc[200][3], sach=14, trich="200 QPS | 2 | 1.4 | 14% | 14%")
theo_sach("500 QPS: batch trung bình", cong_thuc[500][0], sach=5, trich="500 QPS | 5 | 2.2 | 0.7% | 3%")
theo_sach("500 QPS: độ lệch chuẩn", cong_thuc[500][1], sach=2.2, nguon="500 QPS · 2.2")
theo_sach("500 QPS: cửa sổ rỗng (%)", cong_thuc[500][2], sach=0.7, trich="500 QPS | 5 | 2.2 | 0.7% | 3%")
theo_sach("500 QPS: batch ≥ 2 lần trung bình (%)", cong_thuc[500][3], sach=3, trich="500 QPS | 5 | 2.2 | 0.7% | 3%")
theo_sach("1000 QPS: batch trung bình", cong_thuc[1000][0], sach=10, trich="1000 QPS | 10 | 3.2 | 0.005% | 0.3%")
theo_sach("1000 QPS: độ lệch chuẩn", cong_thuc[1000][1], sach=3.2, trich="1000 QPS | 10 | 3.2 | 0.005% | 0.3%")
theo_sach("1000 QPS: cửa sổ rỗng (%)", cong_thuc[1000][2], sach=0.005, trich="1000 QPS | 10 | 3.2 | 0.005% | 0.3%")
theo_sach("1000 QPS: batch ≥ 2 lần trung bình (%)", cong_thuc[1000][3], sach=0.3, nguon="1000 QPS · 0.3 %")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(12.5, 3.4))
for ax, lam, mau in zip(axs, MUC_10, MAU):
    m = lam * W10
    k = np.arange(0, int(m + 4 * math.sqrt(m) + 4))
    tan_suat = np.bincount(dem[lam], minlength=len(k))[:len(k)] / len(dem[lam])
    ax.bar(k, tan_suat, color=mau, width=0.8, label="mô phỏng")
    ax.plot(k, [poisson_pmf(int(j), m) for j in k], "o", color="0.15", ms=4, label="phương trình 9")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_title(f"{lam:,} QPS: trung bình {m:g}", fontsize=10)
    ax.set_xlabel("số yêu cầu trong cửa sổ 10 ms")
axs[0].set_ylabel("tỉ lệ cửa sổ")
axs[0].legend(loc="upper right", fontsize=8)
fig.suptitle("Kích thước batch theo cửa sổ lịch cố định: mô phỏng và phương trình 9", fontsize=11)
plt.tight_layout()
plt.show()

Mô phỏng khớp phương trình 9 ở cả bốn mức, và các cột của bảng 10 đều là đọc phân phối Poisson. Ở
50 QPS, gần hai phần ba cửa sổ rỗng, và batch "gấp đôi trung bình" chỉ có nghĩa là một yêu cầu thay
vì không yêu cầu nào. Ở 1,000 QPS, phân phối đã hẹp lại quanh trung bình: lưu lượng cao cho batch
ổn định hơn, đúng như luật số lớn. Ô 1,000 QPS của cột cuối là chỗ duy nhất phép tính sát mép sai số:
chính xác là 0.345 %, sách ghi 0.3 %.

### Batch mà yêu cầu thấy, và đường bao thận trọng

Một yêu cầu không thấy batch trung bình của các cửa sổ. Nó thấy chính nó cộng những yêu cầu khác
cùng đến, nên batch nó thấy là $1 + \text{Poisson}(\lambda_\text{arr} T_\text{window})$, lớn hơn
trung bình một. Phương trình 13 ghép cả cửa sổ với batch ở phân vị 99 mà yêu cầu thấy thành một
đường bao thận trọng:

$$L_\text{lat,envelope} = T_\text{window} + T_\text{svc}(B_\text{req,p99}).$$

**Dự đoán:** p99 thật của độ trễ yêu cầu nằm trên hay dưới đường bao ấy?

In [ ]:
LAM_13 = sach(500, nguon="99 · 500 QPS")
W13 = sach(10, nguon="500 QPS · 10 ms · 12") / 1000
m13 = LAM_13 * W13
B_tb = 1 + m13
B_p99 = 1 + poisson_ppf(0.99, m13)
theo_sach("batch trung bình mà yêu cầu thấy", B_tb, sach=6, trich="the mean request-seen batch size is 6")
theo_sach("batch p99 mà yêu cầu thấy", B_p99, sach=12, nguon="10 ms · 12 · 13.6 ms")
theo_sach("độ trễ trung bình: nửa cửa sổ + T_svc(6) (ms)", 1000 * (W13 / 2 + t_svc(B_tb)), sach=13.6, nguon="12 · 13.6 ms")
theo_sach("T_svc(12) (ms)", 1000 * t_svc(B_p99), sach=12.2, nguon="13.6 ms · 12.2 ms")
bao = W13 + t_svc(B_p99)
theo_sach("đường bao: cả cửa sổ + T_svc(12) (ms)", 1000 * bao, sach=22.2, nguon="12.2 ms · 22.2 ms")
theo_sach("đường bao so với trung bình (lần)", bao / (W13 / 2 + t_svc(B_tb)), sach=1.63, nguon="22.2 ms · 1.63 lần")

tre13, cho13, thay13 = gom_theo_lich(den_poisson(LAM_13, 200_000, np.random.default_rng(42)), W13)
print(f"mô phỏng: batch mà yêu cầu thấy trung bình {thay13.mean():.2f}, p99 {np.percentile(thay13, 99):.0f};"
      f" độ trễ trung bình {1000 * tre13.mean():.2f} ms, p99 {1000 * np.percentile(tre13, 99):.2f} ms,"
      f" lâu nhất {1000 * tre13.max():.1f} ms")
print(f"yêu cầu vượt đường bao: {np.mean(tre13 > bao):.3%}")

Mô phỏng cho cùng batch trung bình và batch p99 với công thức, và độ trễ trung bình gần đúng 13.6 ms.
p99 của độ trễ nằm dưới đường bao: như chương nói, đường bao không phải p99 của độ trễ yêu cầu, vì
yêu cầu chờ hết cửa sổ (tới ngay khi cửa sổ mở) và batch ở p99 không nhất thiết rơi vào cùng một
yêu cầu. Một phần rất nhỏ yêu cầu vẫn vượt đường bao: đó là những lần batch lớn tới mức GPU chưa
xong khi cửa sổ kế tiếp đóng, nên batch sau phải xếp hàng. Đường bao không tính phần hàng đợi ấy.

## 4. Năng lực khi bão hoà

Dưới tải duy trì, việc gom batch sau chồng lên lúc batch trước đang chạy, nên năng lực do thời gian
phục vụ của batch quyết định. Phương trình 10 cho

$$\mu_\text{eff}(B) \approx \frac{B}{T_\text{svc}(B)}, \qquad
\text{thông lượng} = \min(\lambda_\text{arr}, \mu_\text{eff}(B)),$$

với $T_\text{svc}(B) = 5 + 0.6B$ ms cho ResNet-50 trên V100. Bảng 11 cộng thêm mức chờ tối đa của
cửa sổ 10 ms vào độ trễ, còn thông lượng bão hoà chỉ dùng thời gian phục vụ.

In [ ]:
W11 = sach(10, nguon="11 · 10 ms · 5.6 ms")
for B in (1, 4, 8, 16, 32):
    print(f"batch {B:>2}: T_svc {1000 * t_svc(B):5.1f} ms · độ trễ tổng {W11 + 1000 * t_svc(B):5.1f} ms"
          f" · thông lượng {B / t_svc(B):6.0f} ảnh/s")
theo_sach("batch 1, T_svc (ms)", 1000 * t_svc(1), sach=5.6, nguon="5.6 ms · 15.6 ms")
theo_sach("batch 1, độ trễ tổng (ms)", W11 + 1000 * t_svc(1), sach=15.6, nguon="5.6 ms · 15.6 ms")
theo_sach("batch 1, thông lượng (ảnh/s)", 1 / t_svc(1), sach=179, nguon="15.6 ms · 179 ảnh")
theo_sach("batch 4, T_svc (ms)", 1000 * t_svc(4), sach=7.4, trich="4 | 7.4 ms | 17.4 ms | 541 img/s | Moderate")
theo_sach("batch 4, thông lượng (ảnh/s)", 4 / t_svc(4), sach=541, trich="4 | 7.4 ms | 17.4 ms | 541 img/s | Moderate")
theo_sach("batch 8, T_svc (ms)", 1000 * t_svc(8), sach=9.8, nguon="9.8 ms · 19.8 ms")
theo_sach("batch 8, độ trễ tổng (ms)", W11 + 1000 * t_svc(8), sach=19.8, nguon="9.8 ms · 19.8 ms")
theo_sach("batch 8, thông lượng (ảnh/s)", 8 / t_svc(8), sach=816, nguon="19.8 ms · 816 ảnh")
theo_sach("batch 16, T_svc (ms)", 1000 * t_svc(16), sach=14.6, trich="16 | 14.6 ms | 24.6 ms | 1096 img/s | High")
theo_sach("batch 16, thông lượng (ảnh/s)", 16 / t_svc(16), sach=1096, trich="16 | 14.6 ms | 24.6 ms | 1096 img/s | High")
theo_sach("batch 32, T_svc (ms)", 1000 * t_svc(32), sach=24.2, nguon="24.2 ms · 34.2 ms")
theo_sach("batch 32, độ trễ tổng (ms)", W11 + 1000 * t_svc(32), sach=34.2, nguon="24.2 ms · 34.2 ms")
theo_sach("batch 32, thông lượng (ảnh/s)", 32 / t_svc(32), sach=1322, nguon="34.2 ms · 1,322 ảnh")
theo_sach("thông lượng batch 32 so với batch 1 (lần)", (32 / t_svc(32)) / (1 / t_svc(1)), sach=7.4, nguon="1,322 ảnh · 7.4 lần")

Ví dụ 1.7 nối kết quả này với định luật sắt: gom batch chia phụ trội cố định cho nhiều yêu cầu. Gọi
hiệu suất là phần thời gian GPU thật sự tính, tức $0.6B / (5 + 0.6B)$.

In [ ]:
theo_sach("hiệu suất ở batch 1 (%)", 100 * MOI_ANH / t_svc(1), sach=10, trich="Efficiency ≈ 10 percent", tol=1)
theo_sach("phần tính ở batch 32 (ms)", 1000 * MOI_ANH * 32, sach=19.2, nguon="19.2 ms · 79 %")
theo_sach("hiệu suất ở batch 32 (%)", 100 * MOI_ANH * 32 / t_svc(32), sach=79, trich="Efficiency ≈ 79 percent", tol=0.5)
NGUONG_PT = sach(10, nguon="0.6 ms · 10 %") / 100
B_du = next(B for B in range(1, 10_000) if PHU_TROI / t_svc(B) < NGUONG_PT)
print(f"phụ trội cố định xuống dưới {100 * NGUONG_PT:.0f} % tổng thời gian từ batch {B_du} (cần 0.6·B > 9 × 5 ms)")

Ở batch 1, hiệu suất chính xác là 10.7 %, sách làm tròn thành khoảng 10 %. Quy tắc của chương là
tăng batch tới khi phụ trội cố định dưới 10 % tổng thời gian, **hoặc** tới khi SLO không cho chờ
thêm. Với mô hình này, ngưỡng 10 % chỉ tới ở batch 76, nên trong mọi kịch bản của mục này, chính SLO
(hay bộ nhớ) dừng batch trước.

### Tối ưu dưới ràng buộc độ trễ

Khi SLO là ràng buộc chặt nhất, phương trình 12 định nghĩa batch tối đa cho phép theo một xấp xỉ
bậc nhất của độ trễ trung bình:

$$B_\text{max} = \max\left\{B : \frac{T_\text{window}}{2} + T_\text{svc}(B) \le L_\text{lat,target}\right\}.$$

Bảng 12 so hai cấu hình giả định dưới SLO p95 là 50 ms.

In [ ]:
SLO_12 = sach(50, nguon="12 · 50 ms · 5 ms")
W_than_trong = sach(5, nguon="50 ms · 5 ms · 2.5 ms")
W_manh_tay = sach(25, nguon="1,140 ảnh · 25 ms")
theo_sach("thận trọng: chờ trung bình (ms)", W_than_trong / 2, sach=2.5, nguon="5 ms · 2.5 ms")
theo_sach("thận trọng: còn cho suy luận (ms)", SLO_12 - W_than_trong / 2, sach=47.5, nguon="2.5 ms · 47.5 ms")
theo_sach("mạnh tay: chờ trung bình (ms)", W_manh_tay / 2, sach=12.5, nguon="25 ms · 12.5 ms")
theo_sach("mạnh tay: còn cho suy luận (ms)", SLO_12 - W_manh_tay / 2, sach=37.5, nguon="12.5 ms · 37.5 ms")
NL_than_trong = sach(1140, nguon="32 · 1,140 ảnh")      # năng lực bão hoà giả định
NL_manh_tay = sach(1280, nguon="48 · 1,280 ảnh")
theo_sach("năng lực giả định cao hơn (%)", 100 * (NL_manh_tay / NL_than_trong - 1), sach=12.3, nguon="1,280 ảnh · 12.3 %")
theo_sach("chờ trung bình tăng (ms)", (W_manh_tay - W_than_trong) / 2, sach=10, nguon="12.3 % · 10 ms")
theo_sach("mức chờ tối đa tăng (ms)", W_manh_tay - W_than_trong, sach=20, nguon="10 ms · 20 ms")

for ten, W, gioi_han, nl in (("thận trọng", W_than_trong, sach(32, nguon="47.5 ms · 32"), NL_than_trong),
                             ("mạnh tay", W_manh_tay, sach(48, nguon="37.5 ms · 48"), NL_manh_tay)):
    B12 = max(B for B in range(1, 1000) if W / 2 + 1000 * t_svc(B) <= SLO_12)
    print(f"{ten}: phương trình 12 với T_svc = 5 + 0.6B cho B_max = {B12}; bảng 12 giả định giới hạn {gioi_han}"
          f" · phương trình 10 ở batch {gioi_han}: {gioi_han / t_svc(gioi_han):,.0f} ảnh/s, bảng 12 giả định {nl:,}")

Bốn phép trừ và phép chia của bảng 12 khép lại. Nhưng bảng ghi rõ giới hạn batch và năng lực bão hoà
là **giả định**, và ô trên cho thấy chúng không đến từ phương trình 12 và 10 với mô hình 5 + 0.6B:
phương trình 12 cho phép batch lớn hơn nhiều so với 32 và 48, và phương trình 10 cho năng lực cao
hơn 1,140 và 1,280. Trong một hệ thống thật, giới hạn batch còn đến từ bộ nhớ GPU, và năng lực đo
được thường thấp hơn công thức. Sách cũng lưu ý mức chờ tối đa tăng 20 ms không phải một ước lượng
p99.

## 5. Gom batch động và biên Pareto

Bộ lập lịch của phần này là gom batch động: cửa sổ mở khi yêu cầu đầu tiên đang chờ tới, và batch
được gửi khi đủ $B_\text{max}$ yêu cầu hoặc khi hết cửa sổ, tuỳ điều gì đến trước, miễn GPU đã rảnh.
Bảng 13 đưa năm cấu hình (cửa sổ, batch tối đa) với p99 và thông lượng giả định. Ô dưới cho cả năm
cấu hình chạy dưới cùng mô hình phục vụ 5 + 0.6B, ở nhiều mức lưu lượng, rồi đo p99.

**Dự đoán:** hai cấu hình cùng batch tối đa 32, cửa sổ 5 ms và 10 ms. Cái nào chịu được lưu lượng
cao hơn?

In [ ]:
BANG13 = [  # cửa sổ (ms), batch tối đa, p99 giả định (ms), thông lượng giả định (ảnh/s)
    (sach(2, nguon="13 · 2 ms · 16"), sach(16, nguon="2 ms · 16"), sach(18, nguon="16 · 18 ms"),
     sach(890, nguon="18 ms · 890 ảnh")),
    (sach(5, nguon="890 ảnh · 5 ms · 32"), sach(32, nguon="5 ms · 32 · 22 ms"), sach(22, nguon="32 · 22 ms"),
     sach(1140, nguon="22 ms · 1,140 ảnh")),
    (sach(10, trich="10 | 32 | 15 ms | 35 ms | 1,240 img/s | Moderate latency"),
     sach(32, trich="10 | 32 | 15 ms | 35 ms | 1,240 img/s | Moderate latency"), sach(35, nguon="35 ms · 1,240 ảnh"),
     sach(1240, nguon="35 ms · 1,240 ảnh")),
    (sach(20, nguon="1,240 ảnh · 20 ms · 64"), sach(64, nguon="20 ms · 64"), sach(52, nguon="64 · 52 ms"),
     sach(1310, nguon="52 ms · 1,310 ảnh")),
    (sach(50, nguon="1,310 ảnh · 50 ms · 128"), sach(128, nguon="50 ms · 128"), sach(98, nguon="128 · 98 ms"),
     sach(1350, nguon="98 ms · 1,350 ảnh")),
]
theo_sach("thông lượng giả định tăng (%)", 100 * (BANG13[-1][3] / BANG13[0][3] - 1), sach=52, nguon="1,350 ảnh · 52 %")
theo_sach("p99 giả định tăng (lần)", BANG13[-1][2] / BANG13[0][2], sach=5.4, nguon="52 % · 5.4 lần")

LUU_LUONG = np.array([100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1050, 1100, 1150, 1200, 1250, 1300,
                      1350, 1400, 1450, 1500])
duong13 = {}
for W, Bm, _, _ in BANG13:
    tran = Bm / t_svc(Bm)                              # năng lực bão hoà, phương trình 10
    ys = []
    for lam in LUU_LUONG:
        if lam >= 0.97 * tran:                         # sát hoặc quá bão hoà: hàng đợi không ổn định
            ys.append(np.nan)
            continue
        tre_d, _ = gom_dong(den_poisson(lam, 60_000, np.random.default_rng(42)), W / 1000, Bm)
        ys.append(1000 * np.percentile(tre_d, 99))
    duong13[(W, Bm)] = np.array(ys)
    p99_in = " / ".join(f"{y:5.1f}" if np.isfinite(y) else "không ổn định" for y in duong13[(W, Bm)][[1, 7, 13]])
    print(f"cửa sổ {W:>2} ms, batch tối đa {Bm:>3}: năng lực {tran:5.0f} ảnh/s · p99 ở 200 / 800 / 1,200 QPS: {p99_in} ms")

tot_nhat = np.nanmin(np.vstack(list(duong13.values())), axis=0)
for lam in (200, 800, 1000, 1200, 1400):
    i = int(np.where(LUU_LUONG == lam)[0][0])
    ai = [k for k, ys in duong13.items() if ys[i] == tot_nhat[i]][0]
    print(f"{lam:>5} QPS: p99 thấp nhất {tot_nhat[i]:5.1f} ms, thuộc cấu hình cửa sổ {ai[0]} ms, batch {ai[1]}")

p5, p10 = duong13[(5, 32)], duong13[(10, 32)]
co_ca_hai = np.isfinite(p5) & np.isfinite(p10)
print(f"cùng batch tối đa 32: cửa sổ 10 ms có p99 thấp hơn cửa sổ 5 ms ở {np.sum(p10[co_ca_hai] < p5[co_ca_hai])}"
      f" trên {np.sum(co_ca_hai)} mức lưu lượng; chênh lớn nhất {np.max(p10[co_ca_hai] - p5[co_ca_hai]):.1f} ms,"
      f" nhỏ nhất {np.min(p10[co_ca_hai] - p5[co_ca_hai]):.2f} ms")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(LUU_LUONG, tot_nhat, color="0.85", lw=9, solid_capstyle="round", zorder=1,
        label="biên Pareto của mô phỏng")
for (k, ys), mau in zip(duong13.items(), MAU):
    ax.plot(LUU_LUONG, ys, "o-", color=mau, lw=2, ms=4.5, mec="white", zorder=2,
            label=f"cửa sổ {k[0]} ms, batch tối đa {k[1]} (mô phỏng)")
for (W, Bm, p99, tl), mau in zip(BANG13, MAU):
    ax.plot(tl, p99, "D", ms=8, mfc="white", mec=mau, mew=2, zorder=3)
ax.plot([], [], "D", ms=8, mfc="white", mec="0.3", mew=2, label="bảng 13 (giả định của sách)")
ax.set_xlim(0, 1550)
ax.set_ylim(0, 150)
ax.set_xlabel("lưu lượng được phục vụ (yêu cầu mỗi giây)")
ax.set_ylabel("p99 độ trễ (ms)")
ax.set_title("Năm cấu hình gom batch động, mô phỏng với $T_\\mathrm{svc}(B) = 5 + 0.6B$ ms")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Mỗi đường là một cấu hình: p99 thấp ở tải nhẹ, rồi vọt lên khi lưu lượng tiến tới năng lực
$B_\text{max}/T_\text{svc}(B_\text{max})$ của nó, và dừng ở chỗ hàng đợi không còn ổn định. Dải xám
là biên Pareto: ở mỗi mức lưu lượng, cấu hình có p99 thấp nhất. Cấu hình nhỏ thắng ở tải nhẹ vì
nó chờ ít; cấu hình lớn là lựa chọn duy nhất ở tải cao vì chỉ nó còn ổn định. Đó đúng là nội dung
của bảng 13: muốn thêm thông lượng thì phải trả bằng phần đuôi.

Câu trả lời cho dự đoán nằm ở hai đường cùng batch 32: chúng chung một trần, vì trong mô hình này
năng lực chỉ phụ thuộc $B_\text{max}$. Ở tải thấp, cửa sổ 10 ms chậm hơn cửa sổ 5 ms vài mili giây,
vì nó chờ lâu hơn mà không gom thêm được bao nhiêu. Ở tải cao, hai đường gần như trùng nhau, vì batch
đầy trước khi cửa sổ kịp đóng nên độ dài cửa sổ không còn tác dụng; vài mức lưu lượng nghiêng về cửa
sổ 10 ms chỉ do nhiễu của mô phỏng. Không có mức nào mà cửa sổ dài hơn đáng chọn. Bảng 13 cho cấu
hình 10 ms thông lượng cao hơn, nhưng đó là con số giả định; nó chỉ đúng khi thời gian phục vụ thật
được lợi thêm từ batch lớn theo cách mà mô hình bậc nhất không có. Các điểm hình thoi của bảng 13 vì
thế không nằm trên đường của sổ tay, và không cần nằm: sách gọi chúng là minh hoạ.

## 6. Cấu hình gom batch thực tế

Bảng 14 đi ngược từ ngân sách độ trễ: dành 30 % SLO 50 ms cho thời gian chờ gom batch, chọn cửa sổ
dưới mức đó, rồi kiểm đường bao và năng lực.

In [ ]:
SLO_14 = sach(50, nguon="30 % · 50 ms")
PHAN_CHO = sach(30, nguon="30 % · 50 ms · 15 ms") / 100
theo_sach("cửa sổ tối đa (ms)", PHAN_CHO * SLO_14, sach=15, nguon="50 ms · 15 ms · 500 QPS")
LAM_14 = sach(500, nguon="15 ms · 500 QPS")
W14 = sach(12, nguon="32 · 12 ms") / 1000
GIOI_HAN_BO_NHO = sach(32, nguon="13 · 32 · 12 ms")    # trần batch giả định do bộ nhớ GPU
m14 = LAM_14 * W14
theo_sach("batch kỳ vọng mà yêu cầu thấy", 1 + m14, sach=7,
          trich="Expected request-seen batch | 7 | Average batch seen by an arriving request.")
B14_p99 = 1 + poisson_ppf(0.99, m14)
theo_sach("batch p99 mà yêu cầu thấy", B14_p99, sach=13, nguon="500 QPS · 13")
assert B14_p99 <= GIOI_HAN_BO_NHO
theo_sach("đường bao thận trọng (ms)", 1000 * (W14 + t_svc(B14_p99)), sach=24.8, nguon="12 ms · 24.8 ms")
HIEU_SUAT = sach(0.89, nguon="1,176.9 ảnh · 0.89")
theo_sach("năng lực bão hoà minh hoạ (ảnh/s)", HIEU_SUAT * GIOI_HAN_BO_NHO / t_svc(GIOI_HAN_BO_NHO), sach=1176.9,
          nguon="24.8 ms · 1,176.9 ảnh")
theo_sach("thông lượng thực phục vụ (ảnh/s)", min(LAM_14, HIEU_SUAT * GIOI_HAN_BO_NHO / t_svc(GIOI_HAN_BO_NHO)),
          sach=500, nguon="0.89 · 500")

tre14, _ = gom_dong(den_poisson(LAM_14, 200_000, np.random.default_rng(42)), W14, GIOI_HAN_BO_NHO)
print(f"mô phỏng cùng cấu hình với gom batch động: trung bình {1000 * tre14.mean():.1f} ms,"
      f" p99 {1000 * np.percentile(tre14, 99):.1f} ms, so với SLO {SLO_14} ms")

Cấu hình của bảng 14 có biên rộng: đường bao 24.8 ms còn cách xa SLO 50 ms, và mô phỏng với bộ lập
lịch động cho p99 còn thấp hơn đường bao. Điều đó hợp lý ở 500 QPS, mức tải chỉ bằng chưa tới nửa
năng lực. Bước cuối của chương là theo dõi phân phối thật của kích thước batch và thời gian chờ, để
biết giả định Poisson có đúng không.

### Cửa sổ ngắn lại khi lưu lượng tăng

Với yêu cầu đến theo Poisson, phương trình 14 cho số yêu cầu kỳ vọng trong cửa sổ bằng
$\lambda_\text{arr} T_\text{window}$. Phương trình 15 là một quy tắc kinh nghiệm cho độ dài cửa sổ:

$$T_\text{window} \approx \min\left(L_\text{lat,SLO} - T_\text{svc},\ \sqrt{\frac{T_\text{svc}}{\lambda_\text{arr}}}\right).$$

Sách nói dạng căn bậc hai là một quy tắc theo mô hình chi phí cục bộ, không phải lời giải tối ưu.
Bảng 15 tính nó với SLO 50 ms và thời gian phục vụ giả định 25 ms.

In [ ]:
LAM_14b = sach(200, nguon="14 · 200 yêu cầu")
W14b = sach(10, nguon="200 yêu cầu · 10 ms") / 1000
print(f"phương trình 14: {LAM_14b} yêu cầu/s × 10 ms = {LAM_14b * W14b:g} yêu cầu kỳ vọng mỗi cửa sổ")
theo_sach("cửa sổ rỗng ở 200 QPS, 10 ms (%)", 100 * poisson_pmf(0, LAM_14b * W14b), sach=14, nguon="10 ms · 14 %")
print(f"cửa sổ có từ bốn yêu cầu trở lên: {1 - poisson_cdf(3, LAM_14b * W14b):.1%}")

SLO_15 = sach(50, nguon="15 · 50 ms · 25 ms") / 1000
TSVC_15 = sach(25, nguon="50 ms · 25 ms") / 1000


def cua_so_15(lam):
    return min(SLO_15 - TSVC_15, math.sqrt(TSVC_15 / lam))


for lam in (100, 500, 1000, 5000):
    W = cua_so_15(lam)
    print(f"{lam:>5} QPS: cửa sổ {1000 * W:5.2f} ms · batch trung bình {lam * W:5.2f} · độ trễ xấp xỉ {1000 * (W + TSVC_15):5.1f} ms")
theo_sach("100 QPS: cửa sổ (ms)", 1000 * cua_so_15(sach(100, nguon="25 ms · 100 QPS")), sach=15.8, nguon="100 QPS · 15.8 ms")
theo_sach("100 QPS: batch trung bình", 100 * cua_so_15(100), sach=1.6, nguon="15.8 ms · 1.6")
theo_sach("100 QPS: độ trễ xấp xỉ (ms)", 1000 * (cua_so_15(100) + TSVC_15), sach=40.8, nguon="1.6 · 40.8 ms")
theo_sach("500 QPS: cửa sổ (ms)", 1000 * cua_so_15(sach(500, nguon="40.8 ms · 500 QPS")), sach=7.1, nguon="500 QPS · 7.1 ms")
theo_sach("500 QPS: batch trung bình", 500 * cua_so_15(500), sach=3.5, nguon="7.1 ms · 3.5")
theo_sach("500 QPS: độ trễ xấp xỉ (ms)", 1000 * (cua_so_15(500) + TSVC_15), sach=32.1, nguon="3.5 · 32.1 ms")
theo_sach("1,000 QPS: cửa sổ (ms)", 1000 * cua_so_15(sach(1000, nguon="32.1 ms · 1,000 QPS")), sach=5, nguon="1,000 QPS · 5 ms")
theo_sach("1,000 QPS: độ trễ xấp xỉ (ms)", 1000 * (cua_so_15(1000) + TSVC_15), sach=30, nguon="5 ms · 30 ms")
theo_sach("5,000 QPS: cửa sổ (ms)", 1000 * cua_so_15(sach(5000, nguon="30 ms · 5,000 QPS")), sach=2.24, nguon="5,000 QPS · 2.24 ms")
theo_sach("5,000 QPS: batch trung bình", 5000 * cua_so_15(5000), sach=11.2, nguon="2.24 ms · 11.2")
theo_sach("5,000 QPS: độ trễ xấp xỉ (ms)", 1000 * (cua_so_15(5000) + TSVC_15), sach=27.2, nguon="11.2 · 27.2 ms")
for lam in (100, 5000):
    print(f"{lam:>5} QPS: một batch phục vụ mất {TSVC_15 / cua_so_15(lam):.1f} lần độ dài cửa sổ")

Lưu lượng tăng 50 lần, cửa sổ ngắn lại gần 7 lần, mà batch trung bình vẫn tăng, vì lưu lượng cao
lấp batch nhanh hơn. Đó là kết quả trái trực giác mà chương nêu.

Dòng cuối của ô trên nói thêm một điều về cột độ trễ. Nó cộng cửa sổ với thời gian phục vụ, tức coi
như batch không phải chờ batch trước. Với thời gian phục vụ 25 ms mà cửa sổ chỉ 2.24 ms, mỗi batch
chạy hơn mười cửa sổ, nên một GPU duy nhất sẽ để các batch xếp hàng; chúng chỉ không phải chờ khi
nhiều bản sao thay nhau nhận chúng. Sách gọi cột ấy là ngân sách xấp xỉ, không phải p99 đo được, và đây là một lý
do.

## 7. Khi nào chờ gom batch là đáng

Phần này phỏng theo phần C của phòng thí nghiệm đi kèm chương: cùng một dòng yêu cầu đi qua hai
chính sách. Một là batch 1, gửi từng yêu cầu ngay khi GPU rảnh; hai là gom batch động với cửa sổ
5 ms và batch tối đa 32. Thời gian phục vụ vẫn là $5 + 0.6B$ ms, nên batch 1 phục vụ được tối đa
$1/5.6\text{ ms} \approx 179$ yêu cầu mỗi giây. Đầu tiên là lưu lượng đều, ở nhiều mức.

**Dự đoán:** ở 50 yêu cầu mỗi giây, chính sách nào có độ trễ trung bình thấp hơn? Còn p99?

In [ ]:
CHINH_SACH = {"batch 1": (0.0, 1), "gom batch động, 5 ms, tối đa 32": (0.005, 32)}
LAM_DEU = np.array([10, 20, 30, 40, 50, 60, 80, 100, 120, 140, 160, 170])
deu = {ten: [] for ten in CHINH_SACH}
for lam in LAM_DEU:
    den_d = den_poisson(lam, 100_000, np.random.default_rng(42))
    for ten, (W, Bm) in CHINH_SACH.items():
        tre_d, _ = gom_dong(den_d, W, Bm)
        deu[ten].append((1000 * tre_d.mean(), 1000 * np.percentile(tre_d, 99)))
deu = {ten: np.array(v) for ten, v in deu.items()}
b1, dg = deu["batch 1"], deu["gom batch động, 5 ms, tối đa 32"]
for lam, (m1, q1), (m2, q2) in zip(LAM_DEU, b1, dg):
    print(f"{lam:>4} yêu cầu/s: batch 1 trung bình {m1:6.1f} ms, p99 {q1:6.1f} ms · gom batch động trung bình {m2:5.1f} ms, p99 {q2:5.1f} ms")
print(f"gom batch động có trung bình thấp hơn từ {LAM_DEU[np.argmax(dg[:, 0] < b1[:, 0])]} yêu cầu/s,"
      f" p99 thấp hơn từ {LAM_DEU[np.argmax(dg[:, 1] < b1[:, 1])]} yêu cầu/s")

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, j, tieu_de in zip(axs, (0, 1), ("Độ trễ trung bình (mô phỏng)", "p99 (mô phỏng)")):
    for (ten, v), mau in zip(deu.items(), (MAU[1], MAU[0])):
        ax.plot(LAM_DEU, v[:, j], "o-", color=mau, lw=2, ms=5, mec="white", label=ten)
    ax.axvline(1 / t_svc(1), color="0.5", lw=1, ls=":")
    ax.text(1 / t_svc(1) - 3, 92, "năng lực\nbatch 1", ha="right", fontsize=8, color="0.35")
    ax.set_xlabel("lưu lượng đều (yêu cầu mỗi giây)")
    ax.set_title(tieu_de)
    ax.set_xlim(0, 190)
    ax.set_ylim(0, 110)
axs[0].set_ylabel("độ trễ (ms)")
axs[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Ở lưu lượng thấp, batch 1 có độ trễ trung bình thấp hơn: nó không chờ cửa sổ nào, còn gom batch động
trả gần trọn cửa sổ 5 ms mà hiếm khi gom được hơn một yêu cầu. Nhưng p99 đổi chiều sớm hơn nhiều. Với
batch 1, mỗi yêu cầu chiếm GPU 5.6 ms, nên hàng đợi đã hình thành ở mức sử dụng vừa phải, và phần
đuôi chịu nó trước. Gom batch động chia phụ trội 5 ms cho những yêu cầu tới gần nhau, nên hàng đợi
tan nhanh. Câu trả lời cho "khi nào chờ gom batch là đáng" vì thế phụ thuộc thước đo: chấm bằng trung
bình thì gom batch chỉ đáng khi batch 1 đã chạy ở khoảng hai phần ba năng lực, chấm bằng p99 thì đáng
từ sớm hơn nhiều. Đây chính là cạm bẫy đánh giá hệ thống bằng độ trễ trung bình, nhìn từ phía gom batch.

Giờ cho một đợt tăng vọt đi qua cả hai. Mọi thông số lưu lượng là **giả định của sổ tay**: nền 100
yêu cầu mỗi giây, rồi một đợt 400 yêu cầu mỗi giây kéo dài 5 giây, rồi lại 100.

**Dự đoán:** trong đợt tăng, p99 của batch 1 lên tới cỡ nào: vài chục mili giây, hay hàng giây?

In [ ]:
NEN, DOT = 100, 400                                 # yêu cầu mỗi giây, giả định
truoc, dot, sau = 10.0, 5.0, 25.0                   # giây
rng = np.random.default_rng(42)
den_bao = np.concatenate([np.sort(rng.uniform(a, a + d, rng.poisson(lam * d)))
                          for lam, a, d in ((NEN, 0, truoc), (DOT, truoc, dot), (NEN, truoc + dot, sau))])
ket_qua_bao = {}
for ten, (W, Bm) in CHINH_SACH.items():
    tre_b, thay_b = gom_dong(den_bao, W, Bm)
    ket_qua_bao[ten] = (tre_b, thay_b)
    nen = den_bao < truoc
    print(f"{ten:>32}: lúc nền trung bình {1000 * tre_b[nen].mean():5.1f} ms, p99 {1000 * np.percentile(tre_b[nen], 99):5.1f} ms · cả quãng p99"
          f" {1000 * np.percentile(tre_b, 99):8.1f} ms · lâu nhất {1000 * tre_b.max():8.1f} ms"
          f" · kịp 50 ms {np.mean(tre_b <= 0.05):6.1%}")

In [ ]:
CUA = 0.5                                           # giây mỗi ô thời gian
bien = np.arange(0, truoc + dot + sau + CUA, CUA)
o = np.digitize(den_bao, bien) - 1
giua = bien[:-1] + CUA / 2
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
for ax in (ax1, ax2):
    ax.axvspan(truoc, truoc + dot, color="0.92", lw=0)
for (ten, (tre_b, thay_b)), mau in zip(ket_qua_bao.items(), (MAU[1], MAU[0])):
    ax1.plot(giua, [1000 * np.percentile(tre_b[o == k], 99) for k in range(len(giua))], color=mau, lw=2, label=ten)
    ax2.plot(giua, [thay_b[o == k].mean() for k in range(len(giua))], color=mau, lw=2, label=ten)
ax1.axhline(50, color="0.3", lw=1, ls="--")
ax1.text(0.5, 60, "50 ms", fontsize=8, color="0.3")
ax1.text(truoc + dot / 2, 1.5, "đợt tăng\n400 yêu cầu/s", ha="center", va="bottom", fontsize=8, color="0.3")
ax1.set_yscale("log")
ax1.set_ylim(1, 3e4)
ax1.set_ylabel("p99 mỗi 0.5 s (ms, thang log)")
ax1.set_title("Một đợt tăng vọt qua hai chính sách (giả định của sổ tay, mô phỏng)")
ax1.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
ax2.set_ylabel("batch trung bình")
ax2.set_xlabel("thời điểm đến (s)")
ax2.set_xlim(0, truoc + dot + sau)
ax2.set_ylim(0, None)
plt.tight_layout()
plt.show()

Trong đợt tăng, tốc độ đến vượt năng lực của batch 1, phần tồn đọng tích lại, và độ trễ lên hàng
giây, kéo dài rất lâu sau khi đợt tăng đã qua, trong lúc hàng đợi xả dần. Gom batch động thì tự lớn
batch lên khi tải tăng (biểu đồ dưới), nên năng lực của nó tăng theo, và p99 hầu như không đổi.

Gom batch đổi một ít độ trễ trung bình ở tải nhẹ lấy năng lực ở tải nặng, và cửa sổ giới hạn phần
độ trễ phải trả. Đó là lý do chương gọi nó là đòn bẩy cốt lõi của kinh tế phục vụ.

## 8. Hai dạng lưu lượng khác

Gom batch phụ thuộc nhiều vào cách yêu cầu đến. Lưu lượng Poisson của các phần trên là kịch bản
Server của MLPerf. Chương xét thêm hai dạng: nhiều cảm biến đồng bộ, và một người dùng duy nhất.

### Sáu camera, một hạn chót

Ví dụ 1.8 xử lý sáu luồng camera ở 30 khung hình mỗi giây, với hạn chót 33 ms cho mỗi khung. Bảng 16
là dòng thời gian của một bộ khung: camera bắt đầu chụp ở 0 ms, khung của camera 1 tới ở 8 ms, camera
2 tới 5 ở 10 ms, camera 6 tới muộn ở 15 ms. Suy luận batch cho sáu ảnh chạy từ 15 ms tới 25 ms, và kết
quả sẵn sàng cho phần lập kế hoạch ở 32 ms.

In [ ]:
HAN = sach(33, nguon="1.8 · 33 ms")
theo_sach("30 khung hình mỗi giây, mỗi khung (ms)", 1000 / sach(30, nguon="33 ms · 30 khung"), sach=33, nguon="1.8 · 33 ms")
DEN_MUON = sach(15, nguon="10 ms · 15 ms")
XONG_SL = sach(25, nguon="15 ms · 25 ms")
SAN_SANG = sach(32, nguon="25 ms · 32 ms")
SUY_LUAN, SAU_SL = XONG_SL - DEN_MUON, SAN_SANG - XONG_SL
print(f"suy luận batch sáu ảnh: {SUY_LUAN} ms · hậu xử lý và lập kế hoạch: {SAU_SL} ms")
theo_sach("phần dư trong hạn (ms)", HAN - SAN_SANG, sach=1, nguon="32 ms · 1 ms")
bat_dau_muon_nhat = HAN - SUY_LUAN - SAU_SL
print(f"mốc bắt đầu muộn nhất còn kịp hạn: {bat_dau_muon_nhat} ms")

# Mô phỏng biến động (giả định của sổ tay): khung của mỗi camera tới ở 8 ms cộng một độ trễ ngẫu nhiên
# phân phối mũ, trung bình 1.5 ms, độc lập giữa các camera.
SO_CAM = sach(6, trich="Batching 6 camera streams into a single GPU invocation")
rng = np.random.default_rng(42)
toi = sach(8, nguon="0 · 8 ms") + rng.exponential(1.5, size=(100_000, SO_CAM))
cuoi_cung = toi.max(axis=1)
print(f"chờ đủ cả sáu: {np.mean(cuoi_cung + SUY_LUAN + SAU_SL > HAN):.2%} bộ khung trễ hạn")
for cat in (bat_dau_muon_nhat, sach(12, nguon="1 ms · 12 ms")):
    bat_dau = np.minimum(cuoi_cung, cat)             # hết giờ thì bắt đầu với những khung đã có
    dung_lai = np.mean((toi > cat).any(axis=1))
    print(f"hết giờ ở {cat} ms: {np.mean(bat_dau + SUY_LUAN + SAU_SL > HAN):.2%} trễ hạn,"
          f" {dung_lai:.2%} bộ khung dùng lại khung trước của ít nhất một camera,"
          f" xong muộn nhất ở {np.max(bat_dau) + SUY_LUAN + SAU_SL:.1f} ms")

Với 1 ms dư, một phần đáng kể bộ khung trễ hạn khi hệ thống chờ đủ sáu khung, dù biến động giả định
chỉ trung bình 1.5 ms. Một chính sách hết giờ đồng bộ làm số khung trễ hạn về không, đổi lại một phần
bộ khung dùng lại khung trước của camera tới muộn. Sách đặt ngân sách đồng bộ 12 ms và quy tắc dự
phòng dùng lại khung trước, mà không nói ngân sách ấy tính từ mốc nào. Ô trên thử hai mốc hết giờ:
16 ms, mốc muộn nhất mà dòng thời gian của bảng 16 còn cho phép, và 12 ms tính từ lúc camera bắt đầu
chụp, một cách đọc ngân sách của sách. Mốc sớm hơn cho nhiều biên hơn trước hạn chót, đổi lại nhiều
bộ khung phải dùng khung cũ hơn. Bài học không đổi: tải nhiều cảm biến đồng bộ áp đặt hạn chót tất
định, nên pipeline phải có chính sách hết giờ rõ ràng.

### Một người dùng trên điện thoại

Ở đầu kia của phổ, một ứng dụng trên điện thoại xử lý từng yêu cầu một, gần như không có cơ hội gom
batch. Bảng 17 phân tích minh hoạ một lần suy luận thị giác trên điện thoại.

In [ ]:
PHA = {  # pha: (thời gian ms, năng lượng mJ)
    "đọc bộ đệm camera": (sach(8, nguon="17 · 8 ms · 0.08 mJ"), sach(0.08, nguon="8 ms · 0.08 mJ")),
    "giải mã JPEG (CPU)": (sach(15, nguon="0.08 mJ · 15 ms"), sach(1.5, nguon="15 ms · 1.5 mJ")),
    "đổi kích thước, chuẩn hoá": (sach(5, nguon="1.5 mJ · 5 ms"), sach(0.4, nguon="5 ms · 0.4 mJ")),
    "suy luận NPU": (sach(12, nguon="0.4 mJ · 12 ms"), sach(0.8, nguon="12 ms · 0.8 mJ")),
    "hậu xử lý, giao diện": (sach(5, trich="Postprocess + UI | 5 ms | 0.2 mJ | Result rendering"),
                             sach(0.2, nguon="82 % · 0.2 mJ")),
}
tong_ms = sum(t for t, _ in PHA.values())
tong_mJ = sum(e for _, e in PHA.values())
theo_sach("tổng thời gian (ms)", tong_ms, sach=45, nguon="0.2 mJ · 45 ms")
theo_sach("tổng năng lượng (mJ)", tong_mJ, sach=2.98, nguon="45 ms · 2.98 mJ")
theo_sach("khung hình mỗi giây", 1000 / tong_ms, sach=22, nguon="2.98 mJ · 22 khung")
PIN_WH = sach(10, trich="12.1M inferences per 10 Wh battery")
theo_sach("số lần suy luận mỗi pin (triệu)", PIN_WH * 3600 / (tong_mJ / 1000) / 1e6, sach=12.1, nguon="22 khung · 12.1 triệu")
theo_sach("công suất của đường suy luận (mW)", tong_mJ / tong_ms * 1000, sach=66, nguon="12.1 triệu · 66 mW")
CPU_MS = sach(45, trich="CPU fallback replaces the 12 ms, 0.8 mJ NPU inference stage with a 45 ms, 4.2 mJ CPU stage")
CPU_MJ = sach(4.2, nguon="66 mW · 4.2 mJ")
theo_sach("rơi về CPU: tổng thời gian (ms)", tong_ms - PHA["suy luận NPU"][0] + CPU_MS, sach=78, nguon="4.2 mJ · 78 ms")
theo_sach("rơi về CPU: tổng năng lượng (mJ)", tong_mJ - PHA["suy luận NPU"][1] + CPU_MJ, sach=6.4, nguon="78 ms · 6.4 mJ")
for ten, (t, e) in PHA.items():
    print(f"{ten:>26}: {100 * t / tong_ms:4.1f} % thời gian, {100 * e / tong_mJ:4.1f} % năng lượng")
print(f"mức sử dụng ở batch 1: NPU di động {sach(82, nguon='0.8 mJ · 82 %')} %,"
      f" GPU trung tâm dữ liệu của bảng 9 {sach(15, nguon='150 MB · 15 %')} %")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.2), sharey=True)
ten_pha = list(PHA)[::-1]
ax1.barh(ten_pha, [PHA[k][0] for k in ten_pha], color=MAU[0], height=0.6)
ax2.barh(ten_pha, [PHA[k][1] for k in ten_pha], color=MAU[1], height=0.6)
for ax, j, don_vi in ((ax1, 0, "ms"), (ax2, 1, "mJ")):
    for y, k in enumerate(ten_pha):
        ax.text(PHA[k][j], y, f" {PHA[k][j]:g} {don_vi}", va="center", fontsize=8)
    ax.set_xlim(0, max(v[j] for v in PHA.values()) * 1.3)
ax1.set_xlabel("thời gian (ms)")
ax2.set_xlabel("năng lượng (mJ)")
ax1.set_title("Thời gian theo pha (bảng 17)")
ax2.set_title("Năng lượng theo pha (bảng 17)")
plt.tight_layout()
plt.show()

Giải mã JPEG trên CPU chiếm một phần ba thời gian nhưng một nửa năng lượng, nhiều hơn chính bước
suy luận trên NPU. Ở đây NPU chạy ở 82 % mức sử dụng ngay ở batch 1, còn ví dụ trung tâm dữ liệu ở
đầu phần 1 chỉ đạt 15 % ở batch 1. Trên điện thoại, mục tiêu không phải thông lượng tối đa mà là hiệu
năng bền vững: độ trễ chấp nhận được mà không giảm tốc do nhiệt hay hao pin quá mức.

## Thử thêm

1. Ở ô `frontier`, thêm một cấu hình `(5, 64, 0, 0)` vào `BANG13` (hai số cuối chỉ để giữ chỗ). Nó
   đứng ở đâu so với hai cấu hình cùng cửa sổ 5 ms hoặc cùng batch 64? Biên Pareto có đổi không?
2. Viết một cửa sổ thích ứng: trong `gom_dong`, thay `W` cố định bằng một cửa sổ ngắn lại khi hàng đợi
   dài, ví dụ `W_i = W / (1 + so_dang_cho / 8)` với `so_dang_cho` là số yêu cầu đã tới mà chưa được
   gửi lúc yêu cầu `i` tới. Chạy lại ô `burst` với nó. p99 lúc nền và trong đợt tăng đổi ra sao?
3. Ở ô `cameras`, tăng trung bình biến động từ 1.5 lên 3 ms. Ngưỡng hết giờ nào giữ được tỉ lệ dùng
   lại khung dưới 5 % mà vẫn không trễ hạn khung nào?

## Tóm lại

* Gom batch chia hai chi phí cố định, phụ trội mỗi lần gọi và lần đọc trọng số, cho nhiều yêu cầu;
  bảng 9 và phép đo trên CPU có cùng hình dạng: thời gian một batch là $a + bB$, thông lượng tăng
  nhanh rồi phẳng dần.
* Thuế gom batch gồm thời gian chờ và thời gian suy luận phình ra. Gom batch tĩnh khớp phương trình 4
  ở trung bình, nhưng p99 gấp đôi, và ở tải thấp yêu cầu chờ rất lâu.
* Với cửa sổ lịch cố định, chờ trung bình là nửa cửa sổ, chờ tối đa là cả cửa sổ, và kích thước
  batch theo Poisson; đường bao của phương trình 13 nằm trên p99 thật.
* Năng lực khi bão hoà là $B/T_\text{svc}(B)$; trong mô hình bậc nhất nó chỉ phụ thuộc batch tối đa,
  nên cửa sổ dài hơn mà không nâng batch tối đa chỉ thêm độ trễ.
* Biên Pareto đổi phần đuôi lấy thông lượng: cấu hình nhỏ thắng ở tải nhẹ, cấu hình lớn là lựa chọn
  duy nhất ở tải nặng. Gom batch động tự lớn batch khi tải tăng, nên nó sống sót qua đợt tăng vọt mà
  batch 1 không chịu được.
* Lưu lượng đồng bộ cần chính sách hết giờ; lưu lượng một người dùng chuyển mục tiêu từ thông lượng
  sang độ trễ và năng lượng của từng yêu cầu.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Phục vụ mô hình — mỗi yêu cầu một ngân sách độ trễ](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch13-model-serving/), dựng từ chương
[*Model Serving*](https://mlsysbook.ai/vol1/model_serving/model_serving.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_13_model_serving.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.